In [18]:
import pandas as pd #使用pandas读入CSV格式数据
import numpy as np
from pandas import Series,DataFrame

data_train = pd.read_csv("titanic_train.csv")
print(len(data_train.columns))
print(data_train.columns)
# 数据有12列
# 其中Survived为决策属性（Y），其值只有两类1或0，代表着获救或未获救；其他11个属性组成特征属性集合（X）分别是：
# PassengerId（乘客ID），Name（姓名），Ticket（船票信息）##用于预测是否Survived意义不大，可以考虑删除这些属性；
# Pclass（乘客等级），Sex（性别），Embarked（登船港口）是类别型数据；
# Age（年龄），SibSp（堂兄弟妹个数），Parch（父母与小孩的个数），是数值型数据，取值情况较多，但可以转换成类别型数据；
# Fare（票价）是数值型数据；Cabin（船舱）则为文本型数据；
# Age（年龄），Cabin（船舱）和Embarked（登船港口）信息存在缺失数据

# 我们的目标：使用(X,Y)作为训练集，训练一个决策树模型：对于给定的x，预测其y值
# 但实际上个别属性并不能作为输入特征，如Name

12
Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')


In [19]:
print(data_train.describe())#看看数据的分布情况

       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000    0.383838    2.308642   29.699118    0.523008   
std     257.353842    0.486592    0.836071   14.526497    1.102743   
min       1.000000    0.000000    1.000000    0.420000    0.000000   
25%     223.500000    0.000000    2.000000   20.125000    0.000000   
50%     446.000000    0.000000    3.000000   28.000000    0.000000   
75%     668.500000    1.000000    3.000000   38.000000    1.000000   
max     891.000000    1.000000    3.000000   80.000000    8.000000   

            Parch        Fare  
count  891.000000  891.000000  
mean     0.381594   32.204208  
std      0.806057   49.693429  
min      0.000000    0.000000  
25%      0.000000    7.910400  
50%      0.000000   14.454200  
75%      0.000000   31.000000  
max      6.000000  512.329200  


In [20]:
data_train.head()#看看数据的前5行，注意数据中有一些值缺失（NaN）！！（怎么填充它们？）

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [21]:
data_train.info()#注意Age属性、Cabin属性、Embarked属性缺失了很多值！

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [22]:
# 比如年龄Age的缺失，我们可以使用平均值填充
data_train['Age'] = data_train['Age'].fillna(data_train['Age'].median())
# 看看数据的变化
data_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [23]:
# 还有一些值，比如说Sex，取值为字符串：'Male'或'Female'，
# 我们可以将它数值化，比如Male为0，Female为1
data_train.loc[data_train['Sex'] == 'male','Sex'] = 0
data_train.loc[data_train['Sex'] == 'female','Sex'] = 1

# 看看数据的变化
data_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,S


In [24]:
# 但我们可以使用unique函数看看这些属性取值情况，帮助我们做出决策。比如：
print(data_train['Embarked'].unique())
data_train['Embarked'].value_counts()


['S' 'C' 'Q' nan]


Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [25]:
# 还有一些属性如Ticket、Cabin、Embarked等等怎么转换？作为作业留给大家
# Embarked 缺失值根据object类型的特性，使用概率随机数填充（根据出现次数的不同进行改进）
# 统计各类别出现的概率
embarked_probs = data_train['Embarked'].value_counts(normalize=True)
# 随机填充缺失值
na_idx = data_train['Embarked'].isnull()
data_train.loc[na_idx, 'Embarked'] = np.random.choice(embarked_probs.index, size=na_idx.sum(), p=embarked_probs.values)

# Cabin 缺失值与embarked类似，使用概率随机数填充
cabin_probs = data_train['Cabin'].value_counts(normalize=True)
na_cabin_idx = data_train['Cabin'].isnull()
if not cabin_probs.empty:
    data_train.loc[na_cabin_idx, 'Cabin'] = np.random.choice(cabin_probs.index, size=na_cabin_idx.sum(), p=cabin_probs.values)


# Ticket 与Cabin类似，缺失值使用概率随机数填充
ticket_probs = data_train['Ticket'].value_counts(normalize=True)
na_ticket_idx = data_train['Ticket'].isnull()
if not ticket_probs.empty:
    data_train.loc[na_ticket_idx, 'Ticket'] = np.random.choice(ticket_probs.index, size=na_ticket_idx.sum(), p=ticket_probs.values)
#data_train.head()
data_train.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        891 non-null    object 
 11  Embarked     891 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [26]:
# 处理好数据，选择参与计算的属性特征，构造X和Y，就可以使用机器学习算法（如决策树）进行学习建模
# 比如可取Sex、Age作为输入特征X，预测Survived（Y）

X = data_train[['Sex', 'Age']].values
print(X.shape)
print(X[:20])
Y = data_train[['Survived']].values
print(Y.shape)
print(Y[:20])

(891, 2)
[[0 22.0]
 [1 38.0]
 [1 26.0]
 [1 35.0]
 [0 35.0]
 [0 28.0]
 [0 54.0]
 [0 2.0]
 [1 27.0]
 [1 14.0]
 [1 4.0]
 [1 58.0]
 [0 20.0]
 [0 39.0]
 [1 14.0]
 [1 55.0]
 [0 2.0]
 [0 28.0]
 [1 31.0]
 [1 28.0]]
(891, 1)
[[0]
 [1]
 [1]
 [1]
 [0]
 [0]
 [0]
 [0]
 [1]
 [1]
 [1]
 [1]
 [0]
 [0]
 [0]
 [1]
 [0]
 [1]
 [0]
 [1]]


In [28]:
data_train.to_csv('titanic_train_processed.csv', index=False, encoding='utf-8-sig')
data_train["Survived"].to_csv('titanic_train_Y.csv', index=False, encoding='utf-8-sig')
data_train[["Sex", "Age"]].to_csv('titanic_train_X.csv', index=False, encoding='utf-8-sig')